In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches

from IPython.display import HTML
%matplotlib inline
import PIL.Image as Image
import gymnasium as gym
import random
import torch 
import time
import imageio
import datetime
from argparse import Namespace
import pickle 

import ot
import torch

from itertools import count
from torch.utils.tensorboard import SummaryWriter
from collections import deque
from datetime import datetime
from tqdm.notebook import tqdm

import stable_baselines3 as sb
from stable_baselines3 import PPO as PPOSB
from stable_baselines3.common.callbacks import CallbackList, BaseCallback, EvalCallback
from stable_baselines3.common.utils import configure_logger
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from envs.envs2d import CMovePND

/Users/iov/.miniforge3/envs/mldev/lib/python3.9/site-packages/flatbuffers/compat.py:19: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  import imp
/Users/iov/.miniforge3/envs/mldev/lib/python3.9/site-packages/keras/protobuf/versions_pb2.py:18: DeprecationWarning: Call to deprecated create function FileDescriptor(). Note: Create unlinked descriptors is going to go away. Please use get/find descriptors from generated code or query the descriptor_pool.
  DESCRIPTOR = _descriptor.FileDescriptor(
/Users/iov/.miniforge3/envs/mldev/lib/python3.9/site-packages/keras/protobuf/versions_pb2.py:36: DeprecationWarning: Call to deprecated create function FieldDescriptor(). Note: Create unlinked descriptors is going to go away. Please use get/find descriptors from generated code or query the descriptor_pool.
  _descriptor.FieldDescriptor(
/Users/iov/.miniforge3/envs/mldev/lib/python3.9/site-packages/keras/protobuf/versions

In [3]:
def save_traj_dict(env, model, nr_trajectories=10, 
                   min_total_rew=-1e6, deterministic=True, save_eval=None, seed=13):
    num_steps = 0
    expert_obs = []
    expert_acs = []
    expert_rew = []
    expert_done = []

    if isinstance(nr_trajectories, list):
        nr_trajectories = nr_trajectories[0]

    cnt = 0
    ep_rew = []
    while cnt < nr_trajectories:
        ob, _ = env.reset(seed=seed + np.random.randint(0,1000))
        done = False
        total_reward = 0
        episode_obs = []
        episode_acs = []
        episode_rew = []
        episode_done = []

        while not done:
            with torch.no_grad():
                action, _states = model.predict(ob, deterministic=deterministic)
            next_ob, reward, term, trunc, _ = env.step(action)
            done = term or trunc
            ob = next_ob
            total_reward += reward
            episode_obs.append(ob)
            episode_acs.append(action)
            episode_rew.append(reward)
            episode_done.append(done)

            num_steps += 1

        ep_rew.append(total_reward)
        print("episode:", cnt, "reward:", total_reward)
        if total_reward > min_total_rew:
            cnt += 1
            expert_obs.extend(episode_obs)
            expert_acs.extend(episode_acs)
            expert_rew.extend(episode_rew)
            expert_done.extend(episode_done)

    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"{ts}_model_{nr_trajectories}.pkl" 
    expert_obs = np.stack(expert_obs)
    expert_acs = np.stack(expert_acs)
    expert_rew = np.stack(expert_rew)
    expert_done = np.stack(expert_done)

    ep_rew = np.array(ep_rew)

    print("Mean / std over {nr_trajectories} episodes: ", np.mean(ep_rew), np.std(ep_rew))

    d = {'obs': expert_obs, 'acs': expert_acs, 'rew': expert_rew, 'done': expert_done}
    if save_eval is not None:
        pickle.dump(d, open(save_eval, 'wb'))

    return np.mean(ep_rew), np.std(ep_rew), d

def train_expert(env, n_timesteps, kwargs, fname="", save_deterministic=False):
    model_dir = f"./results/{datetime.now().strftime('%Y%m%d_%H%M%S')}_ndnav"
    model = PPOSB('MlpPolicy', env, verbose=0, **kwargs, tensorboard_log=model_dir)
    model.learn(total_timesteps=n_timesteps, callback=[TqdmCallback()])
    
    # save model
    if not os.path.exists(model_dir):
        os.mkdir(model_dir)
        
    model.save(os.path.join(model_dir, 'model_' + fname + str(n_timesteps)))
    mu, std, d = save_traj_dict(env, model, nr_trajectories=10, deterministic=save_deterministic,
                  save_eval = f"{model_dir}/demo.pkl")
    
    return model, d

class TqdmCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.progress_bar = None
    
    def _on_training_start(self):
        self.progress_bar = tqdm(total=self.locals['total_timesteps'])
    
    def _on_step(self):
        self.progress_bar.update(1)
        return True

    def _on_training_end(self):
        self.progress_bar.close()
        self.progress_bar = None


In [4]:
DIM = 2
SIZE = 50
env = CMovePND(waypoints = [np.array([SIZE-1]*DIM)], D=DIM, size=SIZE, n_obstacles = 10)

# default PPO settings?
kwargs = {
    'learning_rate':0.0003, 
    'n_steps':256, 
    'batch_size':64, 
    'n_epochs':3, 
    'gamma':0.99, 
    'gae_lambda':0.95, 
    'clip_range':0.2, 
    'clip_range_vf':None, 
    'ent_coef':0.0, 
    'vf_coef':0.5, 
    #'max_grad_norm':0.5
}
train_exp = True
if train_exp:
    expert_policy, demo = train_expert(env, 100000, kwargs)
else:
    demo = pickle.load(open('results/20231107_001240_ndnav/demo.pkl','rb'))

  0%|          | 0/100000 [00:00<?, ?it/s]

episode: 0 reward: 453.4586112193688
episode: 1 reward: 448.2053433569464
episode: 2 reward: 455.38652942562703
episode: 3 reward: 461.2686541717336
episode: 4 reward: 452.33496063900793
episode: 5 reward: 445.5089699468341
episode: 6 reward: 448.9605556407066
episode: 7 reward: 439.30689819516545
episode: 8 reward: 447.4865473525283
episode: 9 reward: 455.94002979795835
Mean / std over {nr_trajectories} episodes:  450.78570997458763 5.89620479297746


### Prepare data

In [5]:
expert_obs = demo['obs']

policy_obs = []
env.reset()
for i in range(len(expert_obs)):
    o,r,t,tr,i = env.step(env.action_space.sample())
    policy_obs.append(o)
    
policy_obs = np.array(policy_obs)

In [6]:
x1 = policy_obs.astype(np.float32)
x2 = expert_obs.astype(np.float32)
fig, ax = plt.subplots(figsize=(8, 8))
dim0 = 0
dim1 = 1
ax.scatter(x1[:, dim0], x1[:, dim1], alpha=0.5, label='$\mu^{(0)}$')
ax.scatter(x2[:, dim0], x2[:, dim1], alpha=0.5, label=r'$\nu$')
for i, j in env.obstacles:
    rect = patches.Rectangle((i[dim0],i[dim1]), j[dim0], j[dim1], linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)
    #plt.vline([1], i[1], j[1])
plt.title('Sliced Wasserstein gradient flow')
plt.legend()
plt.axis('equal')
plt.show()

<>:6: DeprecationWarning: invalid escape sequence \m
<>:6: DeprecationWarning: invalid escape sequence \m
/var/folders/g3/q3xhtj8s5bjgf2p8_8yjnr5c0000gn/T/ipykernel_47244/1492299251.py:6: DeprecationWarning: invalid escape sequence \m
  ax.scatter(x1[:, dim0], x1[:, dim1], alpha=0.5, label='$\mu^{(0)}$')
/var/folders/g3/q3xhtj8s5bjgf2p8_8yjnr5c0000gn/T/ipykernel_47244/1492299251.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Calculate sliced Wasserstein gradient flow

In [7]:
#x1 = policy_obs[:,:2].astype(np.float32)
#x2 = expert_obs[:,:2].astype(np.float32)
x1 = policy_obs.astype(np.float32)
x2 = expert_obs.astype(np.float32)

device = "cuda" if torch.cuda.is_available() else "cpu"

# use pyTorch for our data
x1_torch = torch.tensor(x1).to(device=device).requires_grad_(True)
x2_torch = torch.tensor(x2).to(device=device)

lr = DIM*50
NPROJ = 100
N = len(x1)
nb_iter_max = 1000
OT_MODE = 'sliced'

x_all = np.zeros((nb_iter_max, *x1_torch.shape))

loss_iter = []

# generator for random permutations
gen = torch.Generator()
gen.manual_seed(42)

for i in tqdm(range(nb_iter_max)):

    # unbalanced optimal transport?
    if OT_MODE == 'sliced':    
        #loss = ot.sliced_wasserstein_distance(x1_torch, x2_torch, n_projections=NPROJ, seed=gen)
        loss = ot.max_sliced_wasserstein_distance(x1_torch, x2_torch, seed=gen)
    else:
        a, b = torch.ones((N,)) / N, torch.ones((N,)) / N  # uniform distribution on samples
        M = ot.dist(x1_torch, x2_torch)
        lambd = 1e-1

        #loss = ot.sinkhorn(a, b, M, lambd)
        loss = ot.bregman.empirical_sinkhorn2(x1_torch, x2_torch, reg=lambd)
        #loss = ot.unbalanced.sinkhorn_unbalanced2(x1_torch, x2_torch, M, reg=lambd, reg_m=lambd, seed=gen)

    loss_iter.append(loss.clone().detach().cpu().numpy())
    loss.backward()

    # performs a step of PROJECTED gradient descent: TODO: action-conditional projections?
    # TODO: controlled diffusion and Hamiltonian flows
    with torch.no_grad():
        grad = x1_torch.grad
        x1_torch -= grad * lr / (1 + i / 5e1)  # step
        #x1_torch -= simple_proj_on_A(grad) # project onto action space!
        x1_torch.grad.zero_()
        #x1_torch = torch.round(x1_torch)
        x_all[i, :, :] = x1_torch.clone().detach().cpu().numpy()

xb = x1_torch.clone().detach().cpu().numpy()

  0%|          | 0/1000 [00:00<?, ?it/s]

### Try fitting symplectic ODE which respects? Fitting Neural OD

In [8]:
plt.figure(2, (8, 8))
dim0 = 0
dim1 = 1
ax = plt.gca()
plt.scatter(x1[:, dim0], x1[:, dim1], alpha=0.5, label='$\mu^{(0)}$')
plt.scatter(x2[:, dim0], x2[:, dim1], alpha=0.5, label=r'$\nu$')
plt.scatter(xb[:, dim0], xb[:, dim1], alpha=0.5, label='$\mu^{(100)}$')
for i, j in env.obstacles:
    rect = patches.Rectangle((i[dim0],i[dim1]), j[dim0], j[dim1], linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)
plt.title('Sliced Wasserstein gradient flow')
plt.legend()
plt.axis('equal')
plt.show()

<>:5: DeprecationWarning: invalid escape sequence \m
<>:7: DeprecationWarning: invalid escape sequence \m
<>:5: DeprecationWarning: invalid escape sequence \m
<>:7: DeprecationWarning: invalid escape sequence \m
/var/folders/g3/q3xhtj8s5bjgf2p8_8yjnr5c0000gn/T/ipykernel_47244/1481980918.py:5: DeprecationWarning: invalid escape sequence \m
  plt.scatter(x1[:, dim0], x1[:, dim1], alpha=0.5, label='$\mu^{(0)}$')
/var/folders/g3/q3xhtj8s5bjgf2p8_8yjnr5c0000gn/T/ipykernel_47244/1481980918.py:7: DeprecationWarning: invalid escape sequence \m
  plt.scatter(xb[:, dim0], xb[:, dim1], alpha=0.5, label='$\mu^{(100)}$')
/var/folders/g3/q3xhtj8s5bjgf2p8_8yjnr5c0000gn/T/ipykernel_47244/1481980918.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
plt.figure(3, (8, 8))

ax = plt.axis()
dim0 = 0
dim1 = 1
## TODO: loss plot
def _update_plot(i):
    plt.clf()
    plt.scatter(x1[:, dim0], x1[:, dim1], alpha=0.3, label='$\mu^{(0)}$')
    plt.scatter(x2[:, dim0], x2[:, dim1], alpha=0.3, label=r'$\nu$')
    plt.scatter(x_all[i, :, dim0], x_all[i, :, dim1], alpha=0.3, label='$\mu^{(100)}$')
    for i, j in env.obstacles:
        rect = patches.Rectangle((i[dim0],i[dim1]), j[dim0], j[dim1], linewidth=2, edgecolor='r', facecolor='none')
        ax_ = plt.gca()
        ax_.add_patch(rect)
    plt.title('Sliced Wasserstein gradient flow Iter. {}'.format(i))
    plt.axis(ax)
    plt.axis('equal')
    return 1

# interval: 5x
ani = animation.FuncAnimation(plt.gcf(), _update_plot, nb_iter_max, interval=20, repeat_delay=2000)
ani.save(f"results/flow_sw_{DIM}.mp4")

<>:9: DeprecationWarning: invalid escape sequence \m
<>:11: DeprecationWarning: invalid escape sequence \m
<>:9: DeprecationWarning: invalid escape sequence \m
<>:11: DeprecationWarning: invalid escape sequence \m
/var/folders/g3/q3xhtj8s5bjgf2p8_8yjnr5c0000gn/T/ipykernel_47244/87440493.py:9: DeprecationWarning: invalid escape sequence \m
  plt.scatter(x1[:, dim0], x1[:, dim1], alpha=0.3, label='$\mu^{(0)}$')
/var/folders/g3/q3xhtj8s5bjgf2p8_8yjnr5c0000gn/T/ipykernel_47244/87440493.py:11: DeprecationWarning: invalid escape sequence \m
  plt.scatter(x_all[i, :, dim0], x_all[i, :, dim1], alpha=0.3, label='$\mu^{(100)}$')
